This notebook is served as the code of Section 4.1 "Comparison with baselines" and Appendix C.1 "Comparison with baselines and beyond".

1. Experiment Name: Comparison experiment under semi-bandit feedback;
2. Dataset: Yelp;
3. Range: $[0,0.1]$;
4. Description: In this notebook, we fix $k=10, m=30$ in the semi-bandit feedback setting.

In [ ]:
import json
import random
import numpy as np
import os
import time
import pandas as pd
from tqdm import tqdm

GLOBAL_SEED = 2026
# -------- Bandit Class --------
class PerturbedBernoulliBandit:
    def __init__(self, mu, noise_scale):
        self.base_mu = np.clip(mu, 0, 0.1)
        self.noise_scale = noise_scale
    
    def pull(self):
        perturbed_mu = np.clip(
            self.base_mu + np.random.normal(0, self.noise_scale),
            0, 0.1
        )
        return np.random.binomial(1, perturbed_mu)

# -------- Dependent Rounding (for EXP3.M) --------
def dep_round(p, k):
    p = np.array(p, dtype=np.float64).copy()
    n = len(p)
    while True:
        undecided = np.where((p > 1e-8) & (p < 1 - 1e-8))[0]
        if len(undecided) < 2: 
            break
        i, j = undecided[:2]
        alpha = min(1 - p[i], p[j])
        beta = min(p[i], 1 - p[j])
        total = alpha + beta
        if total == 0: 
            break
        prob = beta / total
        if np.random.rand() < prob:
            p[i] += alpha
            p[j] -= alpha
        else:
            p[i] -= beta
            p[j] += beta
        p[i] = np.clip(p[i], 0, 1)
        p[j] = np.clip(p[j], 0, 1)
    
    selected = [i for i in range(n) if p[i] >= 0.999]
    if len(selected) != k:
        sorted_indices = np.argsort(p)[::-1][:k]
        selected = sorted_indices.tolist()
    return selected

# -------- our proposed CMOSS algorithm --------
class CMOSS:
    def __init__(self, m, k, delta):
        self.m = m
        self.k = k
        self.delta = delta
        self.T_i = np.zeros(m)          
        self.hat_mu = np.ones(m)       
    
    def select_arms(self):
        unexplored = np.where(self.T_i == 0)[0]
        if len(unexplored) >= self.k:
            return unexplored[:self.k]
        T_safe = np.where(self.T_i == 0, 1e-8, self.T_i)  
        log_input = 1 / (self.delta * T_safe)
        log_input = np.clip(log_input, 1.0, 1e100)        
        rho = np.sqrt(np.log(log_input) / T_safe)
        rho = np.nan_to_num(rho, nan=0.0, posinf=0.0, neginf=0.0)
        bar_mu = np.minimum(self.hat_mu + rho, 1.0)       
        return np.argsort(bar_mu)[-self.k:][::-1]
    
    def update(self, selected, rewards):
        for i, arm in enumerate(selected):
            self.T_i[arm] += 1
            self.hat_mu[arm] += (rewards[i] - self.hat_mu[arm]) / self.T_i[arm]

# -------- CUCB algorithm --------
class CUCB:
    def __init__(self, m, k):
        self.m = m
        self.k = k
        self.T_i = np.zeros(m)       
        self.hat_mu = np.ones(m)     
        self.t = 0                   
    
    def select_arms(self):
        self.t += 1                   
        unexplored = np.where(self.T_i == 0)[0]
        if len(unexplored) >= self.k:
            return unexplored[:self.k]
        T_safe = np.where(self.T_i == 0, np.inf, self.T_i)  
        rho = np.sqrt(3 * np.log(self.t) / (2 * T_safe))    
        bar_mu = np.minimum(self.hat_mu + rho, 1.0)        
        return np.argsort(bar_mu)[-self.k:][::-1]
    
    def update(self, selected, rewards):
        for i, arm in enumerate(selected):    
            self.T_i[arm] += 1
            self.hat_mu[arm] += (rewards[i] - self.hat_mu[arm]) / self.T_i[arm]

# -------- EXP3.M algorithm --------
class EXP3_M:
    def __init__(self, m, k, gamma):
        self.m = m
        self.k = k
        self.gamma = gamma
        self.weights = np.ones(m)
        self.t = 1

    def select_arms(self):
        self._clip_weights()
        total = np.sum(self.weights_clipped) + 1e-8  
        prob = self.k * ((1 - self.gamma) * self.weights_clipped / total + self.gamma / self.m)
        prob = np.clip(prob, 0, 1) 
        return dep_round(prob, self.k)

    def update(self, selected, rewards):
        self._clip_weights()
        total = np.sum(self.weights_clipped) + 1e-8
        prob = self.k * ((1 - self.gamma) * self.weights_clipped / total + self.gamma / self.m)
        prob = np.clip(prob, 1e-8, 1.0)  

        hat_reward = np.zeros(self.m)
        for arm in selected:
            hat_reward[arm] += rewards[selected.index(arm)] / prob[arm]
        
        update_factor = (self.k * self.gamma / self.m) * hat_reward
        update_factor = np.clip(update_factor, -50, 50)  
        
        mask = ~np.isin(np.arange(self.m), self.S0)
        self.weights[mask] *= np.exp(update_factor[mask])
        self.weights = np.clip(self.weights, 1e-100, 1e100)  
        self.t += 1

    def _clip_weights(self):
        sum_weights = np.sum(self.weights)
        threshold_ratio = (1/self.k - self.gamma/self.m)/(1 - self.gamma)
        threshold_val = threshold_ratio * sum_weights
        sorted_weights = np.sort(self.weights)[::-1]

        if sorted_weights[0] >= threshold_val:
            low = 0
            high = len(sorted_weights)
            best_alpha = 0.0
            
            while low < high:
                mid = (low + high) // 2
                sum_rest = sum_weights - np.sum(sorted_weights[:mid])
                denominator = mid*(1 - threshold_ratio)
                
                if denominator <= 0:
                    high = mid
                    continue
                    
                candidate = (threshold_ratio * sum_rest) / denominator
                if mid > 0 and candidate > sorted_weights[mid-1]:
                    low = mid + 1
                else:
                    best_alpha = candidate
                    high = mid
            
            alpha = best_alpha
            self.S0 = np.where(self.weights >= alpha)[0]
            self.weights_clipped = np.minimum(self.weights, alpha)
        else:
            self.S0 = np.array([], dtype=int)
            self.weights_clipped = self.weights.copy()

    
    def _find_alpha(self, threshold_ratio, sum_weights, sorted_weights):
        m = len(sorted_weights)
        prefix_sum = np.cumsum(sorted_weights)
        best_j = 0
        best_alpha = 0.0
        
        left = 1
        right = m
        best_j = 0
        best_alpha = 0.0
        
        while left <= right:
            mid = (left + right) // 2
            if mid == 0:
                sum_rest = sum_weights
            else:
                sum_rest = sum_weights - prefix_sum[mid-1]
            
            denominator = mid * (1 - threshold_ratio)
            if denominator < 1e-8:
                alpha_j = 0.0
            else:
                alpha_j = (threshold_ratio * sum_rest) / denominator
            
            if mid > 0:
                upper_bound = sorted_weights[mid-1] + 1e-8
            else:
                upper_bound = np.inf
            
            if mid < m:
                lower_bound = sorted_weights[mid] - 1e-8
            else:
                lower_bound = -np.inf
            
            valid = (alpha_j <= upper_bound) and (alpha_j > lower_bound)
        
            if valid:
                best_j = mid
                best_alpha = alpha_j
                left = mid + 1
            else:
                right = mid - 1
        
        if best_j > 0:
            sum_rest = sum_weights - (prefix_sum[best_j-1] if best_j-1 >=0 else 0)
            denominator = best_j * (1 - threshold_ratio)
            if denominator < 1e-8:
                best_alpha = 0.0
            else:
                best_alpha = (threshold_ratio * sum_rest) / denominator
            
            upper_bound = sorted_weights[best_j-1] + 1e-8 if best_j > 0 else np.inf
            lower_bound = sorted_weights[best_j] - 1e-8 if best_j < m else -np.inf
            
            if best_alpha <= upper_bound and best_alpha > lower_bound:
                return best_alpha
        
        return 0.0

# -------- HYBRID algorithm --------
EPS = 10E-12
class HYBRID:
    """
    Interface for semi-bandit algorithms.
    """
    
    def __init__(self, dim, m_size, arms):
        self.dim = dim
        self.m_size = m_size
        if m_size is None or m_size < dim / 2:
            self.gamma = 1.0
        else:
            self.gamma = np.sqrt(1.0 / np.log(m_size))
        self.L = np.array([0.0] * self.dim)
        self.x = np.full(self.dim, self.m_size / self.dim)
        self.time_step = 0
        self.bias = 0
        
        self.mean_losses = 1.0 - np.array(arms)
        best_indices = np.argsort(self.mean_losses)[:m_size]
        self.baseline = (self.mean_losses[best_indices] - 0.5).sum()

    def play(self, action):
        feedback = [-1.0 if random.random() > self.mean_losses[i] else 1.0 for i in action]
        regret = (self.mean_losses[action] - 0.5).sum() - self.baseline
        return feedback, regret


    def sample_action(self, x):
        """

        :param x: List[Float], marginal probabilities
        :return: combinatorial action
        """
        # m-set problem
        order = np.argsort(-x)
        included = np.copy(x[order])
        remaining = 1.0 - included
        outer_samples = [w for w in self.split_sample(included, remaining)]
        weights = list(map(lambda z: z[0], outer_samples))
        _, left, right = outer_samples[np.random.choice(len(outer_samples), p=weights)]
        if left == right - 1:
            sample = range(self.m_size)
        else:
            candidates = [i for i in range(left, right)]
            random.shuffle(candidates)
            sample = [i for i in range(left)] + candidates[:self.m_size - left]
        action = [order[i] for i in sample]
        return action

    def split_sample(self, included, remaining):
        """ 

        :param included: remaining marginal probabilities of sampling a coordinate
        :param remaining: remaining marginal probabilities of not sampling a coordinate
        :return: remaining sampling distributions
        """
        prop = 1.0
        left, right = 0, self.dim
        i = self.dim
        while left < right:
            i -= 1
            active = (self.m_size - left) / (right - left)
            inactive = 1.0 - active
            if active == 0 or inactive == 0:
                yield (prop, left, right)
                return
            weight = min(included[right - 1] / active, remaining[left] / inactive)
            yield weight, left, right
            prop -= weight
            assert prop >= -EPS
            included -= weight * active
            remaining -= weight * inactive
            while right > 0 and included[right - 1] <= EPS:
                right -= 1
            while left < self.dim and remaining[left] <= EPS:
                left += 1
            assert right - left <= i
        if prop > 0.0:
            yield (prop, self.m_size, self.m_size + 1)

    def next(self):
        self.time_step += 1.0
        self.learning_rate = self.get_learning_rate(self.time_step)
        self.solve_optimization()
        return self.sample_action(self.x)


    def update(self, action, feedback):
        if len(action):
            self.L[action] += np.divide((np.array(feedback) + 1.0), self.x[action])
        self.L += self.bias
        self.bias = 0


    def solve_optimization(self):
        max_iter = 100
        iteration = 0
        upper = None
        lower = None
        step_size = 1
        while True:
            iteration += 1
            self.x = np.array(
                [self.solve_unconstrained((l + self.bias) * self.learning_rate, x) for l, x in zip(self.L, self.x)])
            f = self.x.sum() - self.m_size
            df = self.hessian_inverse()
            next_bias = self.bias + f / df
            if f > 0:
                lower = self.bias
                self.bias = next_bias
                if upper is None:
                    step_size *= 2
                    if next_bias > lower + step_size:
                        self.bias = lower + step_size
                else:
                    if next_bias > upper:
                        self.bias = (lower + upper) / 2
            else:
                upper = self.bias
                self.bias = next_bias
                if lower is None:
                    step_size *= 2
                    if next_bias < upper - step_size:
                        self.bias = upper - step_size
                else:
                    if next_bias < lower:
                        self.bias = (lower + upper) / 2
            if iteration > max_iter or abs(f) < 100 * EPS:
                break

        assert iteration < max_iter

    def get_learning_rate(self, time):
        return 1.0 / np.sqrt(time)
    
    def solve_unconstrained(self, loss, warmstart):
        x_val, func_val, dif_func_val, dif_x = warmstart, 1.0, float('inf'), 1.0

        while True:
            func_val = loss - 0.5 / np.sqrt(x_val) + self.gamma * (1.0 - np.log(1.0 - x_val))
            dif_func_val = 0.25 / (np.sqrt(x_val) ** 3) + self.gamma / (1.0 - x_val)
            dif_x = func_val / dif_func_val
            if dif_x > x_val:
                dif_x = x_val / 2
            elif dif_x < x_val - 1.0:
                dif_x = (x_val - 1.0) / 2
            if abs(dif_x) < EPS:
                break
            x_val -= dif_x
        return x_val

    def hessian_inverse(self):
        return (1.0 / (0.25 / np.power(self.x, 1.5) + self.gamma / (1.0 - self.x))).sum() * self.learning_rate

# -------- Yelp dataset --------
def load_real_data():
    script_dir = os.getcwd()
    
    with open(os.path.join(script_dir, 'user_preference.txt'), 'r') as f:
        U_file = [json.loads(line) for line in f]
    with open(os.path.join(script_dir, 'arm_info.txt'), 'r') as f:
        A_file = [json.loads(line) for line in f]

    results = []
    for data1, data2 in zip(A_file, U_file):
        preference_v = data2['preference_v']
        fv = data1['fv']
        scalar_value = sum(p[0] * f[0] for p, f in zip(preference_v, fv))
        results.append(scalar_value)

    results = np.array(results)
    results = 0 + 0.1 * (results - np.min(results)) / (np.max(results) - np.min(results))
    
    np.random.seed(GLOBAL_SEED)
    random_results = random.sample(results.tolist(), 30)
    return random_results

# -------- comparison under semi-bandit feedback --------
class RealDataExperimentRunner:
    def __init__(self, mus, noise_scale):
        self.noise_scale = noise_scale
        self.datasets = {'Yelp_30arms': mus}
        self.param_grid = {
            'CMOSS': {'k': [10, 20], 'delta': [0.00001]},
            'CUCB': {'k': [10, 20]},
            'EXP3_M': {'k': [10, 20], 'gamma': [0.01]},
            'HYBRID': {'k': [10, 20]}
        }
        self.data_dir = "results"
        os.makedirs(self.data_dir, exist_ok=True)

    def run(self, n_sim, T):
        conv_data = []
        
        for dataset, mus in self.datasets.items():
            sorted_mus = sorted(mus, reverse=True)
            for algo in ['HYBRID', 'CMOSS', 'CUCB', 'EXP3_M']:
                params_list = self._gen_params(algo)
                for params in params_list:
                    k = params['k']
                    if algo == 'HYBRID':
                        for sim_id in tqdm(range(n_sim), desc=f"{dataset}-{algo}-k{k}"):
                            sim_seed = GLOBAL_SEED + sim_id * 1000 + hash((algo, k)) % 10_000_000
                            random.seed(sim_seed)
                            np.random.seed(sim_seed)
                            
                            bandits = [PerturbedBernoulliBandit(mu, self.noise_scale) for mu in mus]
                            agent = self._init_algo(algo, len(mus), params, arms=mus)
                            
                            cumulative_time = 0.0 
                            current_regret = 0.0
                            
                            for t in range(T):
                                step_start = time.perf_counter()
                                action = agent.next()
                                feedback, regret = agent.play(action)
                                agent.update(action, feedback)
                                step_time = time.perf_counter() - step_start
                                cumulative_time += step_time
                                current_regret += regret

                                record = {
                                    'dataset': dataset,
                                    'algorithm': algo,
                                    'k': k,
                                    't': t + 1,
                                    'regret': current_regret,
                                    'cumulative_time': cumulative_time,
                                    'noise': self.noise_scale,
                                    'sim_id': sim_id
                                }
                                conv_data.append(record)
                    else:
                        best_exp = sum(sorted_mus[:k])
                        
                        for sim_id in tqdm(range(n_sim), desc=f"{dataset}-{algo}-k{k}"):
                            sim_seed = GLOBAL_SEED + sim_id * 1000 + hash((algo, k)) % 10_000_000
                            random.seed(sim_seed)
                            np.random.seed(sim_seed)
                            
                            bandits = [PerturbedBernoulliBandit(mu, self.noise_scale) for mu in mus]
                            agent = self._init_algo(algo, len(mus), params, arms=mus)
                            
                            cumulative_time = 0.0 
                            current_regret = 0.0
                            
                            for t in range(T):
                                step_start = time.perf_counter()
                                arms = agent.select_arms()
                                rewards = [bandits[i].pull() for i in arms]
                                agent.update(arms, rewards)
                                step_time = time.perf_counter() - step_start
                                cumulative_time += step_time    
                                                        
                                exp_reward = sum(bandits[i].base_mu for i in arms)
                                current_regret += (best_exp - exp_reward)
                                
                                record = {
                                        'dataset': dataset,
                                        'algorithm': algo,
                                        'k': k,
                                        't': t + 1,
                                        'regret': current_regret,
                                        'cumulative_time': cumulative_time,
                                        'noise': self.noise_scale,
                                        'sim_id': sim_id
                                }
                                conv_data.append(record)        
        return pd.DataFrame(conv_data)

        
    def save_data(self, conv_df):
        conv_file = f"{self.data_dir}/l_conv_yelp.csv"
        conv_df.to_csv(conv_file, index=False)

        
    def _gen_params(self, algo):
        params = []
        if algo == 'CMOSS':
            for k in self.param_grid[algo]['k']:
                for delta in self.param_grid[algo]['delta']:
                    params.append({'k': k, 'delta': delta})
        elif algo == 'EXP3_M':  
            for k in self.param_grid[algo]['k']:
                for g in self.param_grid[algo]['gamma']:
                    params.append({'k': k, 'gamma': g})
        else:
            for k in self.param_grid[algo]['k']:
                params.append({'k': k})
        return params

    def _init_algo(self, algo, m, params, arms=None):  
        if algo == 'CMOSS': 
            return CMOSS(m=m, k=params['k'], delta=params['delta'])  
        elif algo == 'CUCB':
            return CUCB(m=m, k=params['k'])
        elif algo == 'EXP3_M':
            return EXP3_M(m=m, k=params['k'], gamma=params['gamma'])
        elif algo == 'HYBRID':  
            return HYBRID(dim=m, m_size=params['k'], arms=arms)

if __name__ == "__main__":
    real_mus = load_real_data()
    
    exp = RealDataExperimentRunner(
        mus=real_mus,
        noise_scale=0
    )
    
    n_sim, T =10, 100000
    conv_df= exp.run(n_sim, T)
    exp.save_data(conv_df)

In [ ]:
def process_results(input_path, output_path, target_steps, target_k=[10, 20]):
    df = pd.read_csv(input_path)

    if "k" in df.columns:
        df = df[df["k"].isin(target_k)]
    else:
        raise ValueError("Input CSV does not contain column `k`, cannot split by k.")

    filtered = df[df['t'].isin(target_steps)]
    grouped = filtered.groupby(['algorithm', 'k', 't']).agg({
        'regret': 'mean',
        'cumulative_time': 'mean'
    }).reset_index()
    
    result = grouped.melt(
        id_vars=['algorithm', 'k', 't'],
        value_vars=['regret', 'cumulative_time'],
        var_name='metric',
        value_name='average_value'
    )
    
    result.to_csv(output_path, index=False)
    print(f"Processed results saved to {output_path}")

if __name__ == "__main__":
    config = {
        'input_file': r'results/l_conv_yelp.csv',
        'output_file': r'results/l_summary_results_yelp.csv',
        'target_steps': [100000],
        'target_k': [10, 20]
    }
    
    process_results(
        input_path=config['input_file'],
        output_path=config['output_file'],
        target_steps=config['target_steps'],
        target_k=config['target_k']
    )
